# 17 — Backtest Module Quickstart

Event-study backtest: replay the live signal/stop/exit decision path over history. See the [backtest README](https://github.com/matteolongo/swing_screener/blob/main/src/swing_screener/backtest/README.md) for full documentation.

In [1]:
from __future__ import annotations
import pandas as pd

pd.set_option("display.width", 140)

from swing_screener.data.providers import get_market_data_provider
from swing_screener.backtest.event_study import run_event_study
from swing_screener.backtest.config import BacktestConfig
from swing_screener.execution.guidance import ExecutionConfig

Fetch OHLCV for AAPL and MSFT from 2022 (required for the 260-bar `min_history` filter), then run the event study. The study replays the live signal/stop/exit decision path and returns per-trade R outcomes.

In [2]:
provider = get_market_data_provider()
ohlcv = provider.fetch_ohlcv(["AAPL", "MSFT"], "2022-01-01", "2023-12-31")

cfg = BacktestConfig(
    k_atr=2.0,
    rr_target=2.0,
    execution=ExecutionConfig(pattern_stop_enabled=True),
)
result = run_event_study(ohlcv, tickers=["AAPL", "MSFT"], config=cfg)
print(f"Total trades: {len(result.trades)}")

Failed writing OHLCV cache /home/memphis/projects/swing_screener/.cache/market_data/by_ticker/AAPL__adj=1.parquet: [Errno 13] Permission denied: '/home/memphis/projects/swing_screener/.cache/market_data/by_ticker/.AAPL__adj=1.parquet.tmp-e3414b2e98b9476ca7015cc4dd486bc9'


Failed writing OHLCV cache /home/memphis/projects/swing_screener/.cache/market_data/by_ticker/MSFT__adj=1.parquet: [Errno 13] Permission denied: '/home/memphis/projects/swing_screener/.cache/market_data/by_ticker/.MSFT__adj=1.parquet.tmp-12861e9b601b4344ab46efef1c305aab'


Failed to persist OHLCV cache index /home/memphis/projects/swing_screener/.cache/market_data/by_ticker/index.json: [Errno 13] Permission denied: '/home/memphis/projects/swing_screener/.cache/market_data/by_ticker/.index.json.tmp-b2769936a3994e1fa5698a85fc77dbeb'


Total trades: 24


In [3]:
print(f"Expectancy (R): {result.metrics.expectancy_r:.2f}")
print(f"Win rate: {result.metrics.win_rate:.1%}")
print(f"Profit factor: {result.metrics.profit_factor:.2f}")
print(f"Max drawdown (R): {result.metrics.max_drawdown_r:.2f}")

Expectancy (R): 0.49
Win rate: 54.2%
Profit factor: 3.66
Max drawdown (R): 1.40


In [4]:
for trade in result.trades[:5]:
    print(f"{trade.ticker}: {trade.r_multiple:.2f}R, {trade.bars_held}d, exit={trade.exit_reason}")

AAPL: -0.40R, 12d, exit=exit_signal
AAPL: 1.60R, 20d, exit=time_exit
AAPL: -1.00R, 4d, exit=stop_hit
AAPL: -0.13R, 9d, exit=exit_signal
AAPL: 0.82R, 20d, exit=time_exit
